# 🐝 Multi-Agent Swarm Architecture

## Building a Travel Booking System with Autonomous Agent Handoffs 🤖

This notebook demonstrates the **Swarm pattern** — a decentralized multi-agent architecture where agents can autonomously **hand off conversations** to each other without a central supervisor.

### 🎯 What You'll Learn

1. **🐝 Swarm Architecture** - How agents communicate peer-to-peer using handoff tools
2. **🔧 Domain-Specific Tools** - Creating flight search/booking and hotel search/booking tools
3. **🔀 Handoff Mechanism** - How `create_handoff_tool()` enables agents to transfer conversations
4. **⚡ Stateful Conversations** - Maintaining user reservations and context across handoffs
5. **🧪 Multi-Turn Interactions** - Testing conversations that span multiple agent handoffs

### 🌟 Swarm vs. Supervisor — What's Different?

| Feature | Supervisor Pattern | Swarm Pattern |
|---|---|---|
| **Control** | Centralized (supervisor routes) | Decentralized (agents hand off) |
| **Routing** | Supervisor decides which agent | Each agent decides when to transfer |
| **Scalability** | Supervisor can become a bottleneck | Agents operate independently |
| **Flexibility** | Top-down task assignment | Organic, conversation-driven routing |
| **Best For** | Known task types, structured workflows | Open-ended conversations, customer service |

### 📋 Our Travel Booking System

```
         User Message
              │
    ┌─────────▼──────────┐
    │ Flight Assistant 🛫 │ ←──── handoff ────→ │ Hotel Assistant 🏨 │
    │  • search_flights   │                      │  • search_hotels   │
    │  • book_flight      │                      │  • book_hotel      │
    └────────────────────┘                       └───────────────────┘
```

Each agent can **transfer the conversation** to the other when the user's needs shift from flights to hotels (or vice versa).

---

*💡 **Pro Tip**: The swarm pattern is inspired by OpenAI's Swarm framework — it excels at customer service scenarios where conversations naturally flow between domains.*

![alt text](image-6.png)

## 1. 🔧 Environment Setup & Installation

### 📦 Required Packages

- **`langgraph-swarm`** - Provides `create_swarm()` and `create_handoff_tool()` for peer-to-peer agent coordination
- **`langchain_core`** - Foundation library with message types and runnable interfaces
- **`langgraph`** - Core framework for stateful agent workflows
- **`langchain_openai`** - OpenAI GPT model integration

In [ ]:
%%capture --no-stderr
# 📦 Install the langgraph-swarm package for swarm-based multi-agent coordination
%pip install langgraph-swarm
# 📦 Install/upgrade core LangChain + LangGraph + OpenAI packages
%pip install --quiet -U langchain_core langgraph langchain_openai

### 🔐 API Key Configuration

Both the flight and hotel assistants use OpenAI's GPT-4o for reasoning and tool calling.

In [ ]:
import os, getpass

def _set_env(var: str):
    """Helper to prompt for an environment variable if not already set."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

# 🔑 Prompt for the OpenAI API key (only if not already in environment)
_set_env("OPENAI_API_KEY")
print("✅ API key configured!")

## 2. 📚 Library Imports & Model Setup

### 🔑 Key Imports:
- **`create_handoff_tool`** — Creates a tool that lets one agent transfer the conversation to another
- **`create_swarm`** — Builds the swarm graph from a list of agents
- **`RunnableConfig`** — Provides runtime configuration (like `user_id`) to tool functions
- **`InMemorySaver`** — Checkpointer for conversation continuity within a thread

In [ ]:
# ============================================================
# 📚 Import all required libraries
# ============================================================
import datetime
from collections import defaultdict   # For creating reservation storage with defaults
from typing import Callable           # Type hint for the prompt factory function
from langchain_core.runnables import RunnableConfig  # Carries runtime config (user_id, etc.)
from langchain_openai import ChatOpenAI  # OpenAI GPT model wrapper

from langchain.agents import create_agent          # Creates ReAct-style agents
from langgraph_swarm import create_handoff_tool, create_swarm  # 🐝 Swarm utilities
from langgraph.checkpoint.memory import InMemorySaver  # Short-term conversation memory
from langgraph.store.memory import InMemoryStore       # Long-term cross-thread storage

# Initialize the LLM — GPT-4o handles reasoning and tool calling for both agents
llm = ChatOpenAI(model="gpt-4o")

print("✅ All imports loaded and model initialized!")

## 3. 📊 Mock Data & Reservation Storage

### 🗄️ Setting Up the Backend

Before building agents, we need the data they'll work with. In a production system, these would be real database queries and API calls. Here we use simple in-memory data structures:

- **`RESERVATIONS`** — A `defaultdict` that stores each user's flight and hotel bookings
- **`FLIGHTS`** — Available flights (simulated database)
- **`HOTELS`** — Available hotels (simulated database)

### 💡 Key Design: User-Scoped Data
Each reservation is keyed by `user_id` (from the runtime config), so multiple users can interact with the system simultaneously without data conflicts.

In [ ]:
# ============================================================
# 📊 Mock reservation storage and sample data
# ============================================================

# RESERVATIONS: Per-user storage for booked flights and hotels
# defaultdict auto-creates an empty reservation for new users
RESERVATIONS = defaultdict(lambda: {"flight_info": {}, "hotel_info": {}})

# Tomorrow's date — used as the default flight date
TOMORROW = (datetime.date.today() + datetime.timedelta(days=1)).isoformat()

# Sample flight data — simulates a database of available flights
FLIGHTS = [
    {
        "departure_airport": "BOS",       # Boston Logan
        "arrival_airport": "JFK",         # New York JFK
        "airline": "Jet Blue",
        "date": TOMORROW,                 # Always tomorrow for demo purposes
        "id": "1",                        # Unique flight identifier
    }
]

# Sample hotel data — simulates a database of available hotels
HOTELS = [
    {
        "location": "New York",
        "name": "McKittrick Hotel",
        "neighborhood": "Chelsea",
        "id": "1",                        # Unique hotel identifier
    }
]

print(f"✅ Mock data ready! {len(FLIGHTS)} flight(s), {len(HOTELS)} hotel(s) available.")

## 4. 🛫 Flight Tools

### 🔧 Building the Flight Assistant's Toolkit

Each tool is a Python function that the LLM can call. LangChain automatically converts the function signature and docstring into a tool schema for function calling.

### 🔑 Key Pattern: `RunnableConfig`
Notice that `book_flight` takes a `config: RunnableConfig` parameter. This lets the tool access **runtime configuration** (like `user_id`) to know which user's reservation to update. LangGraph passes this automatically — the LLM doesn't need to provide it.

In [ ]:
# ============================================================
# 🛫 Flight search tool — queries available flights
# ============================================================
def search_flights(departure_airport: str, arrival_airport: str, date: str) -> list[dict]:
    """Search for available flights."""
    # In production, this would query a real flight API
    # Here we return our mock FLIGHTS data regardless of parameters
    return FLIGHTS

# ============================================================
# 🎫 Flight booking tool — reserves a flight for the user
# ============================================================
def book_flight(flight_id: str, config: RunnableConfig) -> str:
    """Book a flight given a flight ID."""
    # Extract the user_id from runtime config — identifies who's booking
    user_id = config["configurable"].get("user_id")
    # Find the flight matching the given ID
    flight = next(f for f in FLIGHTS if f["id"] == flight_id)
    # Store the flight in this user's reservation record
    RESERVATIONS[user_id]["flight_info"] = flight
    return "Successfully booked flight"

print("✅ Flight tools defined!")

## 5. 🏨 Hotel Tools

### 🔧 Building the Hotel Assistant's Toolkit

These tools mirror the flight tools but operate on hotel data. The same `RunnableConfig` pattern is used to scope bookings to individual users.

In [ ]:
# ============================================================
# 🏨 Hotel search tool — queries available hotels by location
# ============================================================
def search_hotels(location: str) -> list[dict]:
    """Search for available hotels in a given location."""
    # In production, this would query a real hotel API (e.g., Booking.com)
    return HOTELS

# ============================================================
# 🔑 Hotel booking tool — reserves a hotel for the user
# ============================================================
def book_hotel(hotel_id: str, config: RunnableConfig) -> str:
    """Book a hotel given a hotel ID."""
    # Extract the user_id from runtime config
    user_id = config["configurable"].get("user_id")
    # Find the hotel matching the given ID
    hotel = next(h for h in HOTELS if h["id"] == hotel_id)
    # Store the hotel in this user's reservation record
    RESERVATIONS[user_id]["hotel_info"] = hotel
    return "Successfully booked hotel"

print("✅ Hotel tools defined!")

## 6. 🔀 Handoff Tools — The Heart of the Swarm Pattern

### 🐝 How Agent Handoffs Work

This is what makes the swarm pattern unique! **Handoff tools** let one agent transfer the active conversation to another agent. When the flight assistant detects that the user wants to book a hotel, it calls `transfer_to_hotel_assistant` and the conversation seamlessly continues with the hotel assistant.

### 🔑 Key Concept: `create_handoff_tool()`
- Creates a special tool that, when called by the LLM, **switches the active agent**
- The `agent_name` must match the `name` parameter of the target agent
- The `description` helps the LLM decide when to use the handoff

### 💡 Why This Matters
Unlike the supervisor pattern (where routing is centralized), here **each agent decides for itself** when to hand off — making the conversation flow more natural.

In [ ]:
# ============================================================
# 🔀 Create handoff tools for agent-to-agent transfers
# ============================================================
# These are special tools that switch the active agent in the swarm.
# When the flight assistant calls transfer_to_hotel_assistant,
# the conversation is seamlessly handed to the hotel assistant.
# ============================================================

# Handoff: Flight → Hotel (when user asks about hotels)
transfer_to_hotel_assistant = create_handoff_tool(
    agent_name="hotel_assistant",                        # Target agent name (must match agent's name)
    description="Transfer user to the hotel-booking assistant."  # Helps LLM decide when to use
)

# Handoff: Hotel → Flight (when user asks about flights)
transfer_to_flight_assistant = create_handoff_tool(
    agent_name="flight_assistant",
    description="Transfer user to the flight-booking assistant."
)

print("✅ Handoff tools created — agents can now transfer conversations!")

## 7. 🧠 Dynamic Prompt Generator (Advanced)

### 📋 Context-Aware System Prompts

This utility creates **dynamic system prompts** that include the user's current reservation data. Each time an agent runs, the prompt is regenerated with the latest reservation state.

### 💡 Why Dynamic Prompts?
Static prompts don't know about the user's context. By injecting reservation data into the system prompt, the agent always knows what the user has already booked — enabling more helpful responses.

> **Note**: This `make_prompt` function is defined here for reference but the agents below use static prompts for simplicity. In production, you'd wire this into the `system_prompt` parameter.

In [ ]:
# ============================================================
# 🧠 Dynamic prompt factory — injects user context into prompts
# ============================================================
# This is a higher-order function: it returns a prompt function
# that LangGraph calls at runtime with the current state and config.
# ============================================================
def make_prompt(base_system_prompt: str) -> Callable[[dict, RunnableConfig], list]:
    """Create a dynamic prompt function that includes user's reservation data."""
    def prompt(state: dict, config: RunnableConfig) -> list:
        # Extract user_id from runtime config to look up their reservations
        user_id = config["configurable"].get("user_id")
        current_reservation = RESERVATIONS[user_id]
        # Build the system prompt by appending user-specific context
        system_prompt = (
            base_system_prompt
            + f"\n\nUser's active reservation: {current_reservation}"
            + f"Today is: {datetime.datetime.now()}"
        )
        # Return the system message + existing conversation messages
        return [{"role": "system", "content": system_prompt}] + state["messages"]
    return prompt

print("✅ Dynamic prompt generator defined!")

## 8. 🤖 Create the Flight and Hotel Assistants

### 🏗️ Assembling the Agent Team

Each agent receives its **domain tools** plus a **handoff tool** to the other agent:

| Agent | Domain Tools | Handoff Tool |
|---|---|---|
| **Flight Assistant** 🛫 | `search_flights`, `book_flight` | `transfer_to_hotel_assistant` |
| **Hotel Assistant** 🏨 | `search_hotels`, `book_hotel` | `transfer_to_flight_assistant` |

### 💡 Notice the Cross-Wiring
The flight assistant has a tool to transfer to the hotel assistant, and vice versa. This creates a **bidirectional handoff** — the conversation can flow freely between the two agents.

In [ ]:
# ============================================================
# 🛫 Create the Flight Assistant agent
# ============================================================
# Tools: search_flights, book_flight, + handoff to hotel assistant
# The handoff tool lets this agent transfer the conversation
# when the user asks about hotels
# ============================================================
flight_assistant = create_agent(
    llm,                                                    # The GPT-4o model
    [search_flights, book_flight, transfer_to_hotel_assistant],  # Domain tools + handoff
    system_prompt="You are a helpful flight booking assistant. You can search for flights and book them for users. You have access to current reservation information and can transfer users to the hotel assistant when needed.",
    name="flight_assistant",                                # Must match handoff tool target name
)

# ============================================================
# 🏨 Create the Hotel Assistant agent
# ============================================================
# Tools: search_hotels, book_hotel, + handoff to flight assistant
# Mirror of the flight assistant — can hand back to flights
# ============================================================
hotel_assistant = create_agent(
    llm,                                                    # Same GPT-4o model
    [search_hotels, book_hotel, transfer_to_flight_assistant],   # Domain tools + handoff
    system_prompt="You are a helpful hotel booking assistant. You can search for hotels and book them for users. You have access to current reservation information and can transfer users to the flight assistant when needed.",
    name="hotel_assistant",                                 # Must match handoff tool target name
)

print("✅ Flight and hotel assistants created!")

## 9. 🐝 Build the Swarm

### ⚡ Assembling the Multi-Agent Swarm

`create_swarm()` takes a list of agents and produces a LangGraph workflow where:
- Each agent is a node in the graph
- Handoff tools create edges between agents
- The `default_active_agent` is the entry point for new conversations
- A checkpointer tracks which agent is currently active across turns

### 🔑 Key Concept: Active Agent Tracking
Unlike the supervisor pattern, the swarm remembers **which agent was last active**. When a follow-up message arrives on the same thread, it routes directly to that agent (not back to the default).

In [ ]:
# ============================================================
# 🐝 Build and compile the swarm
# ============================================================

# Create a checkpointer for conversation memory within a thread
checkpointer = InMemorySaver()

# create_swarm() builds a graph where agents can hand off to each other
# default_active_agent: which agent handles the first message in a new thread
builder = create_swarm(
    [flight_assistant, hotel_assistant],      # All agents in the swarm
    default_active_agent="flight_assistant"   # Entry point for new conversations
)

# Compile with checkpointer to enable multi-turn conversations
# The checkpointer also tracks which agent is currently active
app = builder.compile(checkpointer=checkpointer)

print("✅ Swarm compiled and ready!")

### 🎨 Visualize the Swarm Graph

Notice how the graph differs from the supervisor pattern — there's no central routing node. Instead, agents connect directly to each other through handoff edges.

In [ ]:
# 🎨 Render the swarm graph — notice the bidirectional handoff edges
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

## 10. 🧪 Test the Travel Booking Swarm

### 🚀 Running Multi-Turn Conversations

We'll test the swarm with a two-turn conversation:
1. **Turn 1**: User asks about flights → flight assistant handles it
2. **Turn 2**: User asks about hotels → flight assistant should **hand off** to hotel assistant

This demonstrates the swarm's key capability: **seamless, autonomous agent handoffs** based on conversation context.

### 🔧 Stream Helper Utility
First, let's define a helper function to print streaming output from the graph.

In [ ]:
# ============================================================
# 🔧 Helper function to print streaming output from the graph
# ============================================================
# This utility processes each chunk from stream() and formats
# messages, agent switches, and other updates for readability.
# ============================================================
def print_stream(stream):
    """Helper function to print streaming output from the graph."""
    for chunk in stream:
        if "messages" in chunk:
            # Print each message with its type (human, ai, tool, etc.)
            for message in chunk["messages"]:
                print(f"[{message.type}]: {message.content}")
        elif "agent" in chunk:
            # 🔀 An agent switch happened — show which agent is now active
            print(f"🤖 Active Agent: {chunk['agent']}")
        else:
            # Catch-all for any other state updates
            print(f"📝 Update: {chunk}")
        print("-" * 50)

print("✅ Stream helper defined!")

### 🛫 Turn 1: Flight Search

The user asks about flights from Boston to New York. Since the `default_active_agent` is `flight_assistant`, this request goes directly to the flight agent.

In [ ]:
# ============================================================
# 🛫 Turn 1: User searches for a flight
# ============================================================
# Config includes both thread_id (for conversation memory) and
# user_id (for user-scoped reservation storage)
# ============================================================
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "i am looking for a flight from boston to ny tomorrow"
        }
    ],
}, config)

In [ ]:
# 📋 Print the flight search conversation
# You should see: user message → tool call (search_flights) → flight results
for m in result['messages']:
    m.pretty_print()

### 🏨 Turn 2: Hotel Booking — Testing the Handoff

Now the user asks about a hotel. The flight assistant should recognize this is outside its domain and **hand off** the conversation to the hotel assistant using `transfer_to_hotel_assistant`.

### 🔑 What to Watch For:
- The flight assistant calls the handoff tool
- The active agent switches to `hotel_assistant`
- The hotel assistant handles the search/booking seamlessly

In [ ]:
# ============================================================
# 🏨 Turn 2: User asks about hotels — triggers a handoff!
# ============================================================
# Same thread_id and user_id — continues the previous conversation.
# The flight assistant should detect "hotel" and hand off to
# the hotel assistant automatically.
# ============================================================
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

result = app.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Now I'd like to book a hotel as well."
        }
    ],
}, config)

In [ ]:
# 📋 Print the hotel booking conversation
# You should see the handoff from flight_assistant → hotel_assistant
# followed by hotel search and booking
for m in result['messages']:
    m.pretty_print()

## 📝 Summary & Key Takeaways

### What We Built
A **travel booking swarm** where a flight assistant and hotel assistant autonomously hand off conversations to each other based on user intent — with no central supervisor needed.

### 🔑 Core Concepts Reviewed

| Concept | What We Learned |
|---|---|
| **Swarm Pattern** | Agents communicate peer-to-peer using handoff tools — no central supervisor |
| **`create_handoff_tool()`** | Creates tools that switch the active agent in the swarm |
| **`create_swarm()`** | Builds the graph from agents with their handoff connections |
| **Domain Tools** | Each agent has specialized tools (search, book) for its domain |
| **`RunnableConfig`** | Passes runtime context (user_id) to tools without LLM involvement |
| **Active Agent Tracking** | The swarm remembers which agent was last active across turns |

### 🐝 Swarm Pattern Best Practices
- **Clear domain boundaries** — each agent should have a well-defined specialty
- **Good handoff descriptions** — help the LLM decide when to transfer
- **Bidirectional handoffs** — users should be able to go back and forth
- **Stateful context** — use `RunnableConfig` to pass user-specific data to tools

### 🚀 Next Steps
- Add more agents (car rental, activity booking, restaurant reservations)
- Implement real API integrations (replace mock data with live services)
- Add error handling for failed bookings and edge cases
- Compare with the **Supervisor-Agent** notebook to see the architectural differences!

---

*💡 The swarm pattern excels at **customer service** and **conversational** use cases where the domain naturally shifts during a conversation.*